# Clustering on a handful of names

`resolve_mentions` links proper mentions, then runs average linkage inside blocks. These cells skip the tagger and hand the resolver a few spans with known offsets. The entry text has to contain those spans, because comma apposition reads the characters between them.

Rules in play:

- Same tokens merge, except a cross-honorific pair whose surname belongs to two given names.
- A token subset merges (`Jonathan` inside `Jonathan Harker`) unless the short token is an ambiguous surname.
- `Mrs` plus a surname joins a person only when that given name already has two surnames. That is why Mrs Westenra stays off Lucy, and why Mrs Harker can join Mina.
- Comma apposition merges `Arthur Holmwood, Lord Godalming`. It refuses `Mrs`, `Mr`, or `Miss` when the surnames differ.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "kg").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(ROOT)


In [ ]:
from src.kg.entries import Entry
from src.kg.mentions import Mention
from src.kg.resolve import resolve_mentions

CFG = {
    "auto_merge_threshold": 0.85,
    "review_threshold": 0.60,
    "jaro_block": 0.92,
    "rare_token_max_surfaces": 30,
    "weights": {"jaro": 0.4, "surname": 0.3, "given": 0.2, "honorific": 0.1},
    "male_givens": ["jonathan", "arthur"],
    "female_givens": ["mina", "lucy"],
}


def entry(text: str) -> Entry:
    return Entry(1, 1, "Journal", "Jonathan Harker", "3 May", "1897-05-03", "journal", text, 1)


def mention(surface: str, start: int, end: int, etype: str = "PER") -> Mention:
    return Mention(
        mention_id=f"m-{start}",
        entry_index=1,
        chapter_number=1,
        sent_id=0,
        start=start,
        end=end,
        surface=surface,
        entity_type=etype,
        mention_class="PROP",
        extractor="notebook",
    )


def ids(text: str, spans: list[tuple]) -> list[str]:
    resolved = resolve_mentions(
        [mention(surface, start, end, etype) for surface, start, end, etype in spans],
        [entry(text)],
        CFG,
    )
    return [m.entity_id for m in resolved.mentions]


holmwood = "Arthur Holmwood, Lord Godalming arrived."
print("Holmwood / Godalming", ids(holmwood, [
    ("Arthur Holmwood", 0, 15, "PER"),
    ("Lord Godalming", 17, 31, "PER"),
]))

castle = "Count Dracula left Castle Dracula."
print("Count / Castle", ids(castle, [
    ("Count Dracula", 0, 13, "PER"),
    ("Castle Dracula", 19, 33, "FAC"),
]))

mina = "Mina Murray wrote. Later Mina Harker wrote."
print("Mina Murray / Mina Harker", ids(mina, [
    ("Mina Murray", 0, 11, "PER"),
    ("Mina Harker", 25, 36, "PER"),
]))


## The Harker collision

Add both full names, plus `Mr. Harker` and `Mrs. Harker`. `Mr. Harker` should join Jonathan. `Mrs. Harker` should join Mina, because Mina is already Murray and Harker in this batch. The two clusters should stay apart. `harker` is an ambiguous surname, so the exact-match step skips the cross-honorific pair.


In [ ]:
text = "Jonathan Harker and Mina Murray, later Mina Harker. Mr. Harker met Mrs. Harker."
# Offsets: Jonathan Harker 0:15, Mina Murray 20:31, Mina Harker 39:50, Mr. Harker 52:62, Mrs. Harker 67:78.
print(repr(text[0:15]), repr(text[20:31]), repr(text[39:50]), repr(text[52:62]), repr(text[67:78]))
clusters = ids(text, [
    ("Jonathan Harker", 0, 15, "PER"),
    ("Mina Murray", 20, 31, "PER"),
    ("Mina Harker", 39, 50, "PER"),
    ("Mr. Harker", 52, 62, "PER"),
    ("Mrs. Harker", 67, 78, "PER"),
])
labels = ["Jonathan Harker", "Mina Murray", "Mina Harker", "Mr. Harker", "Mrs. Harker"]
for label, entity_id in zip(labels, clusters):
    print(f"{label:18} {entity_id}")
assert clusters[0] == clusters[3]
assert clusters[1] == clusters[2] == clusters[4]
assert clusters[0] != clusters[1]
print("Harker family stayed split")
